In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# read data
df = pd.read_csv("listings.csv")
df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2992450,https://www.airbnb.com/rooms/2992450,20251107023918,2025-11-07,city scrape,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,NaN,https://a0.muscache.com/pictures/44627226/0e72...,4621559,...,4.56,3.22,3.67,NaN,f,1,1,0,0,0.07
1,3820211,https://www.airbnb.com/rooms/3820211,20251107023918,2025-11-07,city scrape,Restored Precinct in Center Sq. w/Parking,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://a0.muscache.com/pictures/prohost-api/H...,19648678,...,4.81,4.82,4.78,NaN,f,5,5,0,0,2.30
2,5651579,https://www.airbnb.com/rooms/5651579,20251107023918,2025-11-07,city scrape,Large studio apt by Capital Center & ESP@,"Spacious studio with hardwood floors, fully eq...",The neighborhood is very eclectic. We have a v...,https://a0.muscache.com/pictures/b3fc42f3-6e5e...,29288920,...,4.88,4.77,4.64,NaN,f,2,1,1,0,2.95
3,6623339,https://www.airbnb.com/rooms/6623339,20251107023918,2025-11-07,city scrape,Center Sq. Loft in Converted Precinct w/ Parking,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://a0.muscache.com/pictures/prohost-api/H...,19648678,...,4.70,4.80,4.72,NaN,f,5,5,0,0,2.62
4,9005989,https://www.airbnb.com/rooms/9005989,20251107023918,2025-11-07,city scrape,"Studio in The heart of Center SQ, in Albany NY",(21 years of age or older ONLY) NON- SMOKING.....,"There are many shops, restaurants, bars, museu...",https://a0.muscache.com/pictures/d242a77e-437c...,17766924,...,4.93,4.87,4.77,NaN,f,1,1,0,0,5.53


In [3]:
cols = [
    "price",
    "accommodates",
    "bedrooms",
    "bathrooms_text",
    "room_type",
    "property_type",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating",
    "host_is_superhost"
]

data = df[cols].copy()
data.head()

,price,accommodates,bedrooms,bathrooms_text,room_type,property_type,minimum_nights,number_of_reviews,review_scores_rating,host_is_superhost
0,$70.00,4,2.0,1 bath,Entire home/apt,Entire rental unit,28,9,3.56,f
1,$86.00,3,1.0,1 bath,Entire home/apt,Entire rental unit,2,314,4.75,t
2,$64.00,2,0.0,1 bath,Entire home/apt,Entire rental unit,2,377,4.52,t
3,$85.00,2,0.0,1 bath,Entire home/apt,Entire rental unit,2,332,4.73,t
4,$85.00,4,1.0,1 bath,Entire home/apt,Entire rental unit,1,623,4.79,f


In [4]:
# convert Price column into int
data["price"] = (
    data["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
data["price"] = pd.to_numeric(data["price"], errors="coerce")

# clean bathrooms column
data["bathrooms"] = data["bathrooms_text"].str.extract(r"(\d+\.?\d*)")
data["bathrooms"] = pd.to_numeric(data["bathrooms"], errors="coerce")
data.drop(columns="bathrooms_text", inplace=True)
data.head()

,price,accommodates,bedrooms,room_type,property_type,minimum_nights,number_of_reviews,review_scores_rating,host_is_superhost,bathrooms
0,70.0,4,2.0,Entire home/apt,Entire rental unit,28,9,3.56,f,1.0
1,86.0,3,1.0,Entire home/apt,Entire rental unit,2,314,4.75,t,1.0
2,64.0,2,0.0,Entire home/apt,Entire rental unit,2,377,4.52,t,1.0
3,85.0,2,0.0,Entire home/apt,Entire rental unit,2,332,4.73,t,1.0
4,85.0,4,1.0,Entire home/apt,Entire rental unit,1,623,4.79,f,1.0


In [5]:
# remove extreme price values
data = data.dropna(subset=["price"])
data = data[data["price"] > 0]
data = data[data["price"] < data["price"].quantile(0.99)]

In [6]:
X = data.drop(columns="price")
y = data["price"]

# define feature types
numeric_features = [
    "accommodates",
    "bedrooms",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating",
    "bathrooms"
]

categorical_features = [
    "room_type",
    "property_type",
    "host_is_superhost"
]

In [7]:
# preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# neural network model
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("mlp", MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

/Users/chen.meixu/miniconda3/envs/ds_env/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('mlp', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [8]:
preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R^2:", round(r2, 4))

MAE: 31.6
RMSE: 50.89
R^2: 0.5105
